# 02-Modeling：客户流失模型训练与评估

本笔记本展示完整的建模流程，与 src/main.py 对应。

In [2]:
import sys
from pathlib import Path

notebook_path = Path.cwd()
project_root = notebook_path.parent.parent
sys.path.insert(0, str(notebook_path.parent)) # 将notebook所在目录的父目录添加到sys.path中
sys.path.insert(0, str(project_root)) # 将项目根目录添加到sys.path中

from shared.src.data_utils import load_config
from shared.src.viz_utils import plot_roc_curve, plot_confusion_matrix

config = load_config(notebook_path.parent / "config.yaml")
config

{'project': {'name': '01-customer-churn',
  'version': '0.1.0',
  'random-seed': 42},
 'paths': {'data-dir': './data',
  'outputs-dir': './outputs',
  'mlruns-dir': './mlruns',
  'cache-dir': './.cache'},
 'data': {'dataset-name': 'aai510-group1/telco-customer-churn',
  'target-col': 'Churn',
  'id-col': 'customerID',
  'test-size': 0.2,
  'stratify': True,
  'shuffle': True},
 'preprocessing': {'missing-placeholder': ' ',
  'numeric-like-cols': ['TotalCharges', 'MonthlyCharges', 'tenure'],
  'binary-cols': ['Partner',
   'Dependents',
   'PhoneService',
   'PaperlessBilling',
   'Churn'],
  'multi-category-cols': ['gender',
   'MultipleLines',
   'InternetService',
   'OnlineSecurity',
   'OnlineBackup',
   'DeviceProtection',
   'TechSupport',
   'StreamingTV',
   'StreamingMovies',
   'Contract',
   'PaymentMethod']},
 'features': {'onehot-max-categories': 5, 'scale-numeric': True},
 'model': {'logistic-regression': {'max_iter': 1000,
   'random_state': 42,
   'n_jobs': -1,
   'clas

In [3]:
# 数据准备
from src.data_prep import clean_data, get_feature_target_split, load_data_from_hf
from src.features import build_preprocessor, encode_target, identify_column_types
from sklearn.model_selection import train_test_split

df_raw = load_data_from_hf(config["data"]["dataset-name"])
df_clean = clean_data(df_raw, config)
X, y = get_feature_target_split(df_clean, config)
y = encode_target(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

col_types = identify_column_types(X_train, config)
preprocessor = build_preprocessor(col_types, config)

X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

print(f"训练集: {X_train_p.shape}, 测试集: {X_test_p.shape}")

Generating train split:   0%|          | 0/4225 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1409 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1409 [00:00<?, ? examples/s]

清洗后仍存在缺失值的列:
Churn Category    3104
Churn Reason      3104
Internet Type      886
Offer             2324
dtype: int64


训练集: (3380, 61), 测试集: (845, 61)


In [4]:
# 基线模型：Logistic Regression
from src.models import build_baseline_model
from src.evaluate import evaluate_model

baseline = build_baseline_model(config)
baseline.fit(X_train_p, y_train)

base_metrics = evaluate_model(baseline, X_test_p, y_test, model_name="baseline", output_dir=project_root / "outputs")
base_metrics

/home/zlzhangn/projects/ml-cases-practice/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/zlzhangn/projects/ml-cases-practice/.venv/lib/python3.13/site-packages/seaborn/utils.py:61: UserWarning: Glyph 26410 (\N{CJK UNIFIED IDEOGRAPH-672A}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/home/zlzhangn/projects/ml-cases-practice/.venv/lib/python3.13/site-packages/seaborn/utils.py:61: UserWarning: Glyph 27969 (\N{CJK UNIFIED IDEOGRAPH-6D41}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/home/zlzhangn/projects/ml-cases


=== 分类报告 ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       621
           1       1.00      1.00      1.00       224

    accuracy                           1.00       845
   macro avg       1.00      1.00      1.00       845
weighted avg       1.00      1.00      1.00       845


=== 混淆矩阵 ===
[[621   0]
 [  0 224]]


/home/zlzhangn/projects/ml-cases-practice/shared/src/viz_utils.py:38: UserWarning: Glyph 20551 (\N{CJK UNIFIED IDEOGRAPH-5047}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight", dpi=150)
/home/zlzhangn/projects/ml-cases-practice/shared/src/viz_utils.py:38: UserWarning: Glyph 27491 (\N{CJK UNIFIED IDEOGRAPH-6B63}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight", dpi=150)
/home/zlzhangn/projects/ml-cases-practice/shared/src/viz_utils.py:38: UserWarning: Glyph 29575 (\N{CJK UNIFIED IDEOGRAPH-7387}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight", dpi=150)
/home/zlzhangn/projects/ml-cases-practice/shared/src/viz_utils.py:38: UserWarning: Glyph 30495 (\N{CJK UNIFIED IDEOGRAPH-771F}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight", dpi=150)
/home/zlzhangn/projects/ml-cases-practice/shared/src/viz_utils.py:38: UserWarning: Glyph 26354 (\N{CJK UNIFIED IDEOGRAPH-66F2}) missing from font(s) Dej

{'accuracy': 1.0,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'auc': np.float64(1.0),
 'auc_manual': np.float64(1.0)}

In [5]:
# 主力模型：LightGBM
from src.models import build_main_model

main_model = build_main_model(config)
main_model.fit(X_train_p, y_train)

main_metrics = evaluate_model(main_model, X_test_p, y_test, model_name="lightgbm", output_dir=project_root / "outputs")
main_metrics


=== 分类报告 ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       621
           1       1.00      1.00      1.00       224

    accuracy                           1.00       845
   macro avg       1.00      1.00      1.00       845
weighted avg       1.00      1.00      1.00       845


=== 混淆矩阵 ===
[[621   0]
 [  0 224]]


/home/zlzhangn/projects/ml-cases-practice/.venv/lib/python3.13/site-packages/seaborn/utils.py:61: UserWarning: Glyph 26410 (\N{CJK UNIFIED IDEOGRAPH-672A}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/home/zlzhangn/projects/ml-cases-practice/.venv/lib/python3.13/site-packages/seaborn/utils.py:61: UserWarning: Glyph 27969 (\N{CJK UNIFIED IDEOGRAPH-6D41}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/home/zlzhangn/projects/ml-cases-practice/.venv/lib/python3.13/site-packages/seaborn/utils.py:61: UserWarning: Glyph 22833 (\N{CJK UNIFIED IDEOGRAPH-5931}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/home/zlzhangn/projects/ml-cases-practice/shared/src/viz_utils.py:38: UserWarning: Glyph 26410 (\N{CJK UNIFIED IDEOGRAPH-672A}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight", dpi=150)
/home/zlzhangn/projects/ml-cases-practice/shared/src/viz_utils.py:38: UserWarning: Glyph 27969 (\N{CJK UNIFIED IDEOGRAPH-6D41}) missing from font(s) DejaVu Sa

{'accuracy': 1.0,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'auc': np.float64(1.0),
 'auc_manual': np.float64(1.0)}

In [7]:
# SHAP 可解释性分析
from src.evaluate import shap_analysis

shap_importance = shap_analysis(
    main_model, X_test_p,
    feature_names=list(X_test_p.columns),
    sample_size=500,
    output_dir=project_root / "outputs",
    model_name="lightgbm"
)

print("Top 5 关键特征:")
print(shap_importance.head(5))

/home/zlzhangn/projects/ml-cases-practice/.venv/lib/python3.13/site-packages/shap/explainers/_tree.py:583: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/home/zlzhangn/projects/ml-cases-practice/shared/src/viz_utils.py:38: UserWarning: Glyph 29305 (\N{CJK UNIFIED IDEOGRAPH-7279}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight", dpi=150)
/home/zlzhangn/projects/ml-cases-practice/shared/src/viz_utils.py:38: UserWarning: Glyph 24449 (\N{CJK UNIFIED IDEOGRAPH-5F81}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight", dpi=150)
/home/zlzhangn/projects/ml-cases-practice/shared/src/viz_utils.py:38: UserWarning: Glyph 37325 (\N{CJK UNIFIED IDEOGRAPH-91CD}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight", dpi=150)
/home/zlzhangn/projects/ml-cases-practice/shared/src/viz_utils.py:38: UserWarning: Glyph 35201 (\N{CJK UNIFIED IDEOGRAPH-8981}) m

Top 5 关键特征:
Customer Status_Churned    6.546643e+00
Churn Category             1.655378e+00
Churn Reason               1.968408e-01
Age                        3.156688e-14
Avg Monthly GB Download    9.511204e-15
dtype: float64
